# 🍎 PyTorch Deep Learning Experiment

Welcome to the PyTorch edition of the MLB Pitch Bot! 

In this notebook, we'll build a Neural Network from scratch using **PyTorch**. This is a more manual process than `scikit-learn`, but it gives us total control over the architecture.

## What we'll cover:
1. **Tensors & Data Loaders**: How to get Pandas data into PyTorch.
2. **The Model**: Defining a Multi-Layer Perceptron (MLP).
3. **The Training Loop**: Loss functions, Optimizers, and Backpropagation.
4. **Evaluation**: Comparing it back to our simpler models.

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report

# Setup paths for project imports
root = Path("/Users/michael/Documents/Data Projects/mlb-pitch-bot")
if str(root) not in sys.path:
    sys.path.append(str(root))

from src.database import query_all_pitches
from src.train_model import prepare_target_and_features

print(f"PyTorch Version: {torch.__version__}")

## 1. Data Preparation
Neural networks work with **Tensors**. We need to scale our data and convert it into PyTorch-friendly formats.

In [ ]:
df = query_all_pitches()
X, y, le, _, feature_cols, _, _, _, _, weights = prepare_target_and_features(df, include_batter_stats=False)

# Standard Scaling (Vital for Neural Networks)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Train/Test split
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

# Convert to PyTorch Tensors
X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.long)
X_test_t = torch.tensor(X_test, dtype=torch.float32)
y_test_t = torch.tensor(y_test, dtype=torch.long)

# Create DataLoaders (helps with batching data)
train_ds = TensorDataset(X_train_t, y_train_t)
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)

print(f"Prepared {len(X_train_t)} training samples with {X_train_t.shape[1]} features.")

## 2. Define the Neural Network Architecture
We create a class that inherits from `nn.Module`. We'll use two hidden layers with **ReLU** activation. 

The final layer has 3 outputs (one for each pitch family: Fastball, Breaking, Offspeed).

In [ ]:
class PitchFamilyNN(nn.Module):
    def __init__(self, input_dim, num_classes):
        super(PitchFamilyNN, self).__init__()
        self.layer1 = nn.Linear(input_dim, 64)
        self.layer2 = nn.Linear(64, 32)
        self.output = nn.Linear(32, num_classes)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.2) # Helps prevent overfitting
        
    def forward(self, x):
        x = self.relu(self.layer1(x))
        x = self.dropout(x)
        x = self.relu(self.layer2(x))
        x = self.output(x) # Output raw 'logits'
        return x

model = PitchFamilyNN(input_dim=X_train_t.shape[1], num_classes=len(le.classes_))
print(model)

## 3. The Training Loop
In PyTorch, we have to manually loop over the data, calculate the error (Loss), and tell the optimizer to adjust the weights.

In [ ]:
criterion = nn.CrossEntropyLoss() # Standard for multi-class classification
optimizer = optim.Adam(model.parameters(), lr=0.001)

epochs = 20
for epoch in range(epochs):
    model.train()
    total_loss = 0
    for batch_X, batch_y in train_loader:
        # 1. Clear gradients
        optimizer.zero_grad()
        
        # 2. Forward pass
        outputs = model(batch_X)
        loss = criterion(outputs, batch_y)
        
        # 3. Backward pass (Backpropagation)
        loss.backward()
        
        # 4. Update weights
        optimizer.step()
        
        total_loss += loss.item()
        
    if (epoch + 1) % 5 == 0:
        print(f"Epoch [{epoch+1}/{epochs}], Loss: {total_loss/len(train_loader):.4f}")

print("Training Complete.")

## 4. Evaluation
Let's see how our manually-coded Neural Network compares to the others.

In [ ]:
model.eval()
with torch.no_grad(): # Don't calculate gradients during evaluation
    test_outputs = model(X_test_t)
    _, predicted = torch.max(test_outputs, 1)
    
    print("\n--- PyTorch Neural Network Performance ---")
    print(classification_report(y_test_t.numpy(), predicted.numpy(), target_names=le.classes_))